In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset    

In [2]:
RANDOM_STATE = 42
MAX_ITER = 1000
N_ESTIMATORS = 100
torch.manual_seed(42)
data_path = "C:/GitHub/telecom_churn_predictor/data/processed_data.csv"
df = pd.read_csv(data_path)

In [3]:
X = df.drop("churn_probability", axis=1)
y = df["churn_probability"]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp)
print(f"Split done: Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Before resampling: {y_train.value_counts()}")

Split done: Train: (41999, 136), Val: (14000, 136), Test: (14000, 136)
Before resampling: churn_probability
0    37720
1     4279
Name: count, dtype: int64


In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
print(f"After resampling: {pd.Series(y_train_resampled).value_counts()}")

After resampling: churn_probability
1    37720
0    37720
Name: count, dtype: int64


In [5]:
lr = LogisticRegression(max_iter=MAX_ITER, random_state=RANDOM_STATE)
lr.fit(X_train_resampled, y_train_resampled)

y_val_pred = lr.predict(X_val_scaled)
y_val_proba = lr.predict_proba(X_val_scaled)[:, 1]
print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred))
print("Validation ROC AUC:", roc_auc_score(y_val, y_val_proba))

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.79      0.88     12573
           1       0.32      0.84      0.46      1427

    accuracy                           0.80     14000
   macro avg       0.65      0.82      0.67     14000
weighted avg       0.91      0.80      0.83     14000

Validation ROC AUC: 0.8900449684981961


In [6]:
rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)
rf.fit(X_train_resampled, y_train_resampled)

y_val_pred_rf = rf.predict(X_val_scaled)
y_val_proba_rf = rf.predict_proba(X_val_scaled)[:, 1]
print("Validation Classification Report (Random Forest):")
print(classification_report(y_val, y_val_pred_rf))
print("Validation ROC AUC (Random Forest):", roc_auc_score(y_val, y_val_proba_rf))

Validation Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.97      0.95      0.96     12573
           1       0.62      0.72      0.67      1427

    accuracy                           0.93     14000
   macro avg       0.79      0.84      0.81     14000
weighted avg       0.93      0.93      0.93     14000

Validation ROC AUC (Random Forest): 0.9318409639771011


In [7]:
xgb = XGBClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, eval_metric='logloss')
xgb.fit(X_train_resampled, y_train_resampled)

y_val_pred_xgb = xgb.predict(X_val_scaled)
y_val_proba_xgb = xgb.predict_proba(X_val_scaled)[:, 1]
print("Validation Classification Report (XGBoost):")
print(classification_report(y_val, y_val_pred_xgb))
print("Validation ROC AUC (XGBoost):", roc_auc_score(y_val, y_val_proba_xgb))

Validation Classification Report (XGBoost):
              precision    recall  f1-score   support

           0       0.96      0.96      0.96     12573
           1       0.68      0.66      0.67      1427

    accuracy                           0.93     14000
   macro avg       0.82      0.81      0.82     14000
weighted avg       0.93      0.93      0.93     14000

Validation ROC AUC (XGBoost): 0.9284756141164332


In [8]:
from models import ChurnPredictorNN
nn_model = ChurnPredictorNN(input_dim=X_train_resampled.shape[1], hidden_layers=[128, 64, 32],
                             output_dim=1)

print(nn_model)

ChurnPredictorNN(
  (network): Sequential(
    (0): Linear(in_features=136, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
    (10): Sigmoid()
  )
)


In [9]:
X_train_tensor = torch.tensor(X_train_resampled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_resampled.values, dtype=torch.float32).unsqueeze(1)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

loss = nn.BCELoss()
optimizer = torch.optim.Adam(nn_model.parameters(), lr=0.001)

In [10]:
EPOCHS = 30
for epoch in range(EPOCHS):
    nn_model.train()
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = nn_model(X_batch)
        batch_loss = loss(outputs, y_batch)
        batch_loss.backward()
        optimizer.step()
        epoch_loss += batch_loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {epoch_loss/len(train_loader):.4f}")


Epoch 1/30, Loss: 0.3728
Epoch 2/30, Loss: 0.3199
Epoch 3/30, Loss: 0.2988
Epoch 4/30, Loss: 0.2851
Epoch 5/30, Loss: 0.2734
Epoch 6/30, Loss: 0.2678
Epoch 7/30, Loss: 0.2586
Epoch 8/30, Loss: 0.2505
Epoch 9/30, Loss: 0.2461
Epoch 10/30, Loss: 0.2403
Epoch 11/30, Loss: 0.2360
Epoch 12/30, Loss: 0.2307
Epoch 13/30, Loss: 0.2273
Epoch 14/30, Loss: 0.2257
Epoch 15/30, Loss: 0.2215
Epoch 16/30, Loss: 0.2182
Epoch 17/30, Loss: 0.2173
Epoch 18/30, Loss: 0.2130
Epoch 19/30, Loss: 0.2103
Epoch 20/30, Loss: 0.2089
Epoch 21/30, Loss: 0.2081
Epoch 22/30, Loss: 0.2063
Epoch 23/30, Loss: 0.2042
Epoch 24/30, Loss: 0.1983
Epoch 25/30, Loss: 0.2001
Epoch 26/30, Loss: 0.1973
Epoch 27/30, Loss: 0.1928
Epoch 28/30, Loss: 0.1939
Epoch 29/30, Loss: 0.1917
Epoch 30/30, Loss: 0.1907


In [11]:
nn_model.eval()
with torch.no_grad():
    y_val_proba_nn = nn_model(X_val_tensor).squeeze().numpy()
    y_val_pred_nn = (y_val_proba_nn >= 0.5).astype(int)

print("Validation Classification Report (Neural Network):")
print(classification_report(y_val, y_val_pred_nn))
print("Validation ROC AUC (Neural Network):", roc_auc_score(y_val, y_val_proba_nn))

Validation Classification Report (Neural Network):
              precision    recall  f1-score   support

           0       0.97      0.91      0.94     12573
           1       0.50      0.76      0.61      1427

    accuracy                           0.90     14000
   macro avg       0.74      0.84      0.77     14000
weighted avg       0.92      0.90      0.91     14000

Validation ROC AUC (Neural Network): 0.9086549965162107


In [12]:
y_test_pred_rf = rf.predict(X_test_scaled)
y_test_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

print("Test Classification Report (Random Forest):")
print(classification_report(y_test, y_test_pred_rf))
print("Test ROC AUC (Random Forest):", roc_auc_score(y_test, y_test_proba_rf))

Test Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.97      0.95      0.96     12574
           1       0.64      0.71      0.67      1426

    accuracy                           0.93     14000
   macro avg       0.80      0.83      0.82     14000
weighted avg       0.93      0.93      0.93     14000

Test ROC AUC (Random Forest): 0.9272360361582295


In [ ]:
feature_names = X.columns.tolist()
importances = rf.feature_importances_

feature_df = pd.DataFrame({"feature": feature_names, "importance": importances})
feature_df = feature_df.sort_values(by="importance", ascending=False)
print(feature_df.head(10))
print(feature_df[feature_df['feature'].str.contains('trend')])



              feature  importance
71       loc_ic_mou_8    0.069473
89     total_ic_mou_8    0.067704
23   loc_og_t2m_mou_8    0.052161
17      roam_og_mou_8    0.047133
65   loc_ic_t2m_mou_8    0.045988
62   loc_ic_t2t_mou_8    0.039693
32       loc_og_mou_8    0.036059
14      roam_ic_mou_8    0.033845
104  total_rech_amt_8    0.031056
5              arpu_8    0.026220
        feature  importance
134   mou_trend    0.015427
133  arpu_trend    0.009593
135  rech_trend    0.008898


In [14]:
rfe = RFE(estimator=RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE), 
          n_features_to_select=20)
rfe.fit(X_train_resampled, y_train_resampled)

X_val_rfe = rfe.transform(X_val_scaled)
X_test_rfe = rfe.transform(X_test_scaled)

rf_rfe = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)
rf_rfe.fit(rfe.transform(X_train_resampled), y_train_resampled)

y_test_pred_rfe = rf_rfe.predict(X_test_rfe)
y_test_proba_rfe = rf_rfe.predict_proba(X_test_rfe)[:, 1]
print("Test Classification Report (Random Forest with RFE):")
print(classification_report(y_test, y_test_pred_rfe))
print("Test ROC AUC (Random Forest with RFE):", roc_auc_score(y_test, y_test_proba_rfe))

Test Classification Report (Random Forest with RFE):
              precision    recall  f1-score   support

           0       0.97      0.94      0.96     12574
           1       0.59      0.74      0.66      1426

    accuracy                           0.92     14000
   macro avg       0.78      0.84      0.81     14000
weighted avg       0.93      0.92      0.92     14000

Test ROC AUC (Random Forest with RFE): 0.9270958562058755


In [20]:
selected_features = X.columns[rfe.support_].tolist()
print("Selected features after RFE:", selected_features)

results_df = X_test[selected_features].copy()
results_df['actual_churn'] = y_test.values
results_df['predicted_churn'] = y_test_pred_rfe
results_df['churn_probability'] = y_test_proba_rfe

results_df.to_csv('data/churn_predictions.csv', index=False)
print("Predictions saved to data/churn_predictions.csv")

Selected features after RFE: ['arpu_7', 'arpu_8', 'roam_ic_mou_8', 'roam_og_mou_8', 'loc_og_mou_8', 'total_og_mou_8', 'loc_ic_t2t_mou_8', 'loc_ic_t2m_mou_8', 'loc_ic_mou_7', 'loc_ic_mou_8', 'total_ic_mou_8', 'total_rech_num_6', 'total_rech_num_7', 'total_rech_num_8', 'total_rech_amt_8', 'max_rech_amt_8', 'last_day_rch_amt_8', 'aon', 'arpu_trend', 'mou_trend']
Predictions saved to data/churn_predictions.csv


In [21]:
feat_importance_df = pd.DataFrame({
    'feature': selected_features,
    'importance': rf_rfe.feature_importances_
}).sort_values('importance', ascending=False)

feat_importance_df.to_csv('data/feature_importance.csv', index=False)
print(feat_importance_df)

               feature  importance
10      total_ic_mou_8    0.128106
7     loc_ic_t2m_mou_8    0.096799
9         loc_ic_mou_8    0.093844
5       total_og_mou_8    0.075205
3        roam_og_mou_8    0.065714
4         loc_og_mou_8    0.055952
2        roam_ic_mou_8    0.055311
6     loc_ic_t2t_mou_8    0.052833
14    total_rech_amt_8    0.044752
16  last_day_rch_amt_8    0.042002
15      max_rech_amt_8    0.040459
1               arpu_8    0.039212
8         loc_ic_mou_7    0.027852
18          arpu_trend    0.027450
13    total_rech_num_8    0.026672
19           mou_trend    0.026587
0               arpu_7    0.026258
17                 aon    0.026003
11    total_rech_num_6    0.024833
12    total_rech_num_7    0.024157
